# utils

> Formatting utilities for the session management interface

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from datetime import datetime, timezone
from typing import Optional

## Timestamp Parsing

The state store persists timestamps as ISO strings from SQLite's `CURRENT_TIMESTAMP` (format: `"YYYY-MM-DD HH:MM:SS"`). These helpers parse those strings into `datetime` objects for display.

In [ ]:
#| export
def parse_sqlite_timestamp(
    ts:Optional[str] # ISO timestamp string from SQLite, e.g. "2026-04-08 19:11:50"
) -> Optional[datetime]: # Parsed UTC datetime, or None on failure
    """Parse a SQLite CURRENT_TIMESTAMP string into a timezone-aware UTC datetime."""
    if not ts:
        return None
    try:
        dt = datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
        return dt.replace(tzinfo=timezone.utc)
    except (ValueError, TypeError):
        return None

In [ ]:
assert parse_sqlite_timestamp("2026-04-08 19:11:50") == datetime(2026, 4, 8, 19, 11, 50, tzinfo=timezone.utc)
assert parse_sqlite_timestamp(None) is None
assert parse_sqlite_timestamp("") is None
assert parse_sqlite_timestamp("not a date") is None
print(f"Parsed: {parse_sqlite_timestamp('2026-04-08 19:11:50')}")

Parsed: 2026-04-08 19:11:50+00:00


## Relative Time Formatting

For session lists, absolute timestamps add noise — users care about "how long ago" more than precise dates. `format_relative_time` produces short, human-readable deltas like "2 hours ago" or "yesterday", with a fallback to an absolute date for anything older than a month.

In [ ]:
#| export
def format_relative_time(
    ts:Optional[str], # SQLite ISO timestamp string
    now:Optional[datetime]=None # Reference "now" for testing; defaults to real now in UTC
) -> str: # Short human-readable relative time
    """Format a SQLite timestamp as a relative time string like "2 hours ago"."""
    dt = parse_sqlite_timestamp(ts)
    if dt is None:
        return "—"
    reference = now if now is not None else datetime.now(timezone.utc)
    delta = reference - dt
    seconds = int(delta.total_seconds())
    if seconds < 0:
        return "in the future"
    if seconds < 60:
        return "just now"
    if seconds < 3600:
        minutes = seconds // 60
        return f"{minutes} minute{'s' if minutes != 1 else ''} ago"
    if seconds < 86400:
        hours = seconds // 3600
        return f"{hours} hour{'s' if hours != 1 else ''} ago"
    if seconds < 172800:
        return "yesterday"
    if seconds < 604800:
        days = seconds // 86400
        return f"{days} days ago"
    if seconds < 2592000:  # ~30 days
        weeks = seconds // 604800
        return f"{weeks} week{'s' if weeks != 1 else ''} ago"
    # Fallback: absolute date for anything older than a month.
    return dt.strftime("%b %d, %Y")

In [ ]:
# format_relative_time across the whole delta range.
_now = datetime(2026, 4, 8, 12, 0, 0, tzinfo=timezone.utc)

assert format_relative_time("2026-04-08 11:59:30", now=_now) == "just now"
assert format_relative_time("2026-04-08 11:55:00", now=_now) == "5 minutes ago"
assert format_relative_time("2026-04-08 11:00:00", now=_now) == "1 hour ago"
assert format_relative_time("2026-04-08 10:00:00", now=_now) == "2 hours ago"
assert format_relative_time("2026-04-07 11:00:00", now=_now) == "yesterday"
assert format_relative_time("2026-04-04 12:00:00", now=_now) == "4 days ago"
assert format_relative_time("2026-03-25 12:00:00", now=_now) == "2 weeks ago"
# Older than 30 days falls back to absolute date.
assert "Feb" in format_relative_time("2026-02-01 12:00:00", now=_now)
# None / invalid.
assert format_relative_time(None) == "—"
assert format_relative_time("") == "—"
# Future timestamps (clock skew edge case).
assert format_relative_time("2026-04-08 13:00:00", now=_now) == "in the future"

print(f"5m ago: {format_relative_time('2026-04-08 11:55:00', now=_now)}")
print(f"2h ago: {format_relative_time('2026-04-08 10:00:00', now=_now)}")
print(f"Older: {format_relative_time('2026-02-01 12:00:00', now=_now)}")

5m ago: 5 minutes ago
2h ago: 2 hours ago
Older: Feb 01, 2026


## Absolute Datetime Formatting

Used for tooltip display and detail views where the exact timestamp matters.

In [ ]:
#| export
def format_absolute_datetime(
    ts:Optional[str] # SQLite ISO timestamp string
) -> str: # Formatted datetime, e.g. "Apr 08, 2026 19:11"
    """Format a SQLite timestamp as an absolute human-readable datetime."""
    dt = parse_sqlite_timestamp(ts)
    if dt is None:
        return "—"
    return dt.strftime("%b %d, %Y %H:%M")

In [ ]:
assert format_absolute_datetime("2026-04-08 19:11:50") == "Apr 08, 2026 19:11"
assert format_absolute_datetime(None) == "—"
assert format_absolute_datetime("garbage") == "—"
print(f"Absolute: {format_absolute_datetime('2026-04-08 19:11:50')}")

Absolute: Apr 08, 2026 19:11


## Size Formatting

Session state blobs can range from a few hundred bytes (fresh session) to hundreds of KB (mid-workflow with many segments). A compact size hint helps users spot runaway state.

In [ ]:
#| export
def format_bytes(
    n:Optional[int] # Size in bytes
) -> str: # Compact human-readable size, e.g. "1.2 KB"
    """Format a byte count as a compact human-readable size."""
    if n is None or n < 0:
        return "—"
    if n < 1024:
        return f"{n} B"
    if n < 1024 * 1024:
        return f"{n / 1024:.1f} KB"
    return f"{n / (1024 * 1024):.1f} MB"

In [ ]:
assert format_bytes(0) == "0 B"
assert format_bytes(512) == "512 B"
assert format_bytes(1024) == "1.0 KB"
assert format_bytes(1536) == "1.5 KB"
assert format_bytes(325721) == "318.1 KB"  # the real fixture session size
assert format_bytes(2 * 1024 * 1024) == "2.0 MB"
assert format_bytes(None) == "—"
assert format_bytes(-1) == "—"
print(f"325721 B -> {format_bytes(325721)}")

325721 B -> 318.1 KB


## Default Label Generation

When a session has no stored label and no host label generator is provided, this produces a generic fallback based on the session's created_at timestamp.

In [ ]:
#| export
def default_label(
    created_at:Optional[str] # SQLite ISO timestamp of session creation
) -> str: # Fallback label, e.g. "Session 2026-04-08 19:11"
    """Produce a fallback session label based on creation time."""
    dt = parse_sqlite_timestamp(created_at)
    if dt is None:
        return "Untitled Session"
    return f"Session {dt.strftime('%Y-%m-%d %H:%M')}"

In [ ]:
assert default_label("2026-04-08 19:11:50") == "Session 2026-04-08 19:11"
assert default_label(None) == "Untitled Session"
assert default_label("garbage") == "Untitled Session"
print(f"Default: {default_label('2026-04-08 19:11:50')}")

Default: Session 2026-04-08 19:11


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()